## Setup

In [ ]:
import pandas as pd  # tables of data

data_path = "/content/drive/My Drive/neet_data.csv"  # where the cleaned data lives on Drive

## Mount Google Drive

In [ ]:
from google.colab import drive  # Colab's Google Drive tool
drive.mount("/content/drive")  # connect Google Drive

## Load the data

In [ ]:
df = pd.read_csv(data_path)  # load the cleaned data into a table

## NEET rate by year and sex, ages 16 to 24, 2013 to 2026

In [ ]:
def wmean(data, var):  # weighted mean of one variable
    d = data.dropna(subset=[var])  # leave out people with a missing value
    return (d[var] * d["asecwt"]).sum() / d["asecwt"].sum()  # sum of value x weight, divided by sum of weights

years = list(range(2013, 2027))  # the years in the table, 2013 to 2026
sample = df[df["year"].isin(years) & ((df["age16_19"] == 1) | (df["age20_24"] == 1))].copy()  # keep 2013 to 2026 and ages 16 to 24

label_for = df.drop_duplicates("female").set_index("female")["sex_label"]  # sex_label for each value of female
columns = {"Total": None, label_for[0]: 0, label_for[1]: 1}  # column names and the female value each one uses: Total, Male, Female

left_out = sample["neet"].isna().groupby(sample["year"]).sum()  # people left out in each year because neet is missing
known = sample.dropna(subset=["neet"])  # keep only people whose neet is known

rate = {}  # weighted NEET rate, one row per year
people = {}  # unweighted respondents, same layout
for year in years:  # loop over the years
    year_data = known[known["year"] == year]  # people in this year
    rate[year] = {}  # start this row of the table
    people[year] = {}  # start this row of the counts
    for column_name, female_value in columns.items():  # loop over Total, Male, Female
        group = year_data if female_value is None else year_data[year_data["female"] == female_value]  # everyone, or one sex
        rate[year][column_name] = wmean(group, "neet") * 100  # weighted percent NEET
        people[year][column_name] = len(group)  # unweighted respondents in this cell

table = pd.DataFrame(rate).T  # dictionary -> table, one row per year
people = pd.DataFrame(people).T  # same layout for the counts

print("People left out because neet is missing, by year (unweighted):")  # left-out title
print(left_out.to_string())  # one number per year
print()  # empty line
print("Table. NEET rate, % of people aged 16 to 24 with a known neet (weighted with asecwt).")  # table title
print(table.to_string(float_format="{:.1f}".format))  # table, 1 decimal
print()  # empty line
print("Check: every year from 2013 to 2026 is in the table:", list(table.index) == years)  # True or False
print("Check: every year has a value in all three columns:", bool(table.notna().all().all()))  # True or False
print()  # empty line
print("Unweighted number of respondents in each cell (neet not missing).")  # counts title
print(people.to_string())  # counts, same layout

**What one cell means.** Take the row 2019 and the column "Female". Among women aged 16 to 24 in the 2019 survey whose NEET status is known, the cell is the percent who were NEET: not employed and not in school. The column "Male" is the same percent for men, and the column "Total" is the same percent for men and women together. People whose NEET status could not be determined are left out, not counted as zero. People are counted by how many people in the country they represent (`asecwt`), so each number describes the population, not just the people surveyed. A cell of 10.0 means 10 out of every 100 such people were NEET. The count table under the table shows how many survey respondents sit behind each cell.